In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import random
import sys

sys.path.append("../../mapqc/")
import mapqc
import plotting

In [ ]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

In [ ]:
query_dataset = "Hajdarovic10X"
# query_dataset = "Huisman_2022"

In [ ]:
data_paths = {
    "Huisman_2022": "../../data/HypoMap/mappings/Huisman_mapped/huisman_2022_rna_mapped_default.h5ad",
    "Hajdarovic10X": "../../data/HypoMap/mappings/hajdarovic_2022_mapped_no_filter.h5ad",
}
path_data = data_paths[query_dataset]

In [ ]:
adata = sc.read_h5ad(path_data)

In [ ]:
adata

In [ ]:
adata.obsm["X_umap_unchanged"] = adata.obsm["X_umap"]
adata.obsm["X_umap"] = adata.obsm["X_redone_umap"]

In [ ]:
adata.obsm["X_spca"].shape

In [ ]:
sorted(adata.obs["Dataset"].unique())

In [ ]:
test = pd.crosstab(adata.obs.Dataset, adata.obs["Age"]).loc[query_dataset]
test[test != 0]

In [ ]:
test = pd.crosstab(adata.obs.Dataset, adata.obs["Sample_ID"]).loc[query_dataset]
test[test != 0]

In [ ]:
test = pd.crosstab(adata.obs.Dataset, adata.obs["Sample_ID"]).loc["Dowsett10xnuc"]
test[test != 0]

In [ ]:
# adata.obs['reference_or_query'] = 'reference'
# adata.obs.loc[adata.obs.Dataset == 'Huisman_2022','reference_or_query'] = 'query'

In [ ]:
query_dataset

In [ ]:
if query_dataset == "Huisman_2022":
    ref_q_key = "query_or_reference"
    ref_cat = "reference"
    query_cat = "query"
    condition_key = "disease_or_healthy"
    batch_key = "Dataset"
    sample_key = "Sample_ID"
    emb_key = "X_spca"
    coi = "healthy"
elif query_dataset == "Hajdarovic10X":
    adata.obs["query_or_reference"] = "reference"
    adata.obs.loc[adata.obs.Dataset == query_dataset, "query_or_reference"] = "query"
    ref_q_key = "query_or_reference"
    ref_cat = "reference"
    query_cat = "query"
    condition_key = "Age"
    batch_key = "Dataset"
    sample_key = "Sample_ID"
    emb_key = "X_spca"
    coi = "19-24 months"  # "12 weeks"

In [ ]:
n_nhoods = 100
k = 1000

In [ ]:
adata_healthy = adata[
    (adata.obs[ref_q_key] == ref_cat) | (adata.obs[condition_key] == coi), :
].copy()

In [ ]:
adata_healthy.shape

In [ ]:
query_cell_idc = np.where(adata_healthy.obs[ref_q_key] == query_cat)[0].tolist()
random.seed(0)
center_cell_idc = random.sample(query_cell_idc, k=n_nhoods)
center_cells = np.array(adata_healthy.obs.iloc[center_cell_idc, :].index)

In [ ]:
results = mapqc.mapQC_results(
    adata=adata_healthy,
    center_cells=center_cells,
    k=k,
    emb_key=emb_key,
    sample_key=sample_key,
    ref_q_key=ref_q_key,
    ref_cat=ref_cat,
    query_cat=query_cat,
    batch_key=batch_key,
    exclude_same_batch_for_ref_samples=False,  # THIS SHOULD USUALLY BE SET TO TRUE!
    min_n_samples_q=1,
)

In [ ]:
results.calculate_sample_dists_to_ref_samples()

In [ ]:
results.calculate_nhood_statistics()

In [ ]:
results._nhood_filtered.value_counts()

In [ ]:
plotting.plot_umap_significance(results)

In [ ]:
plotting.plot_umap_filtering_info(results)

In [ ]:
sc.pl.umap(adata, color=batch_key)

In [ ]:
plotting.plot_umap_q_ref_sample_shaded(results, include_legend=True)

In [ ]:
sc.pl.umap(
    results.adata,
    color=ref_q_key,
    palette={
        results.ref_cat: "tab:orange",
        results.query_cat: "tab:blue",
    },
    title=f"{query_dataset} (healthy)",
    frameon=False,
)